# 12 — Train MediaPipe Graph + Spatial/Temporal Transformer

This notebook consumes the graph cache produced by notebook 11. The trained path is **Graph Encoder → Spatial Transformer → Temporal Transformer → Classifier**. It logs loss, Top-1, macro precision/recall/F1, stops on validation loss, restores the best checkpoint, and creates comparison plots. The provided Kaggle test split is evaluated only after checkpoint selection.


In [ ]:
#@title Configuration
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
CLASS_COUNT = 50  #@param {type:'integer'}
RUN_NAME = 'mediapipe_graph_spatial_temporal_top50_v1'  #@param {type:'string'}
MAX_EPOCHS = 50  #@param {type:'integer'}
BATCH_SIZE = 16  #@param {type:'integer'}
LEARNING_RATE = 0.0003  #@param {type:'number'}
WEIGHT_DECAY = 0.04  #@param {type:'number'}
LABEL_SMOOTHING = 0.15  #@param {type:'number'}
DROPOUT = 0.40  #@param {type:'number'}
EARLY_STOPPING_MIN_EPOCHS = 10  #@param {type:'integer'}
EARLY_STOPPING_PATIENCE = 7  #@param {type:'integer'}
EARLY_STOPPING_MIN_DELTA = 0.002  #@param {type:'number'}
COPY_GRAPH_CACHE_TO_LOCAL = True  #@param {type:'boolean'}
RESUME = True  #@param {type:'boolean'}
RUN_TEST_AFTER_TRAINING = True  #@param {type:'boolean'}
SEED = 42  #@param {type:'integer'}


In [ ]:
#@title Mount Drive and define artifacts
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_mediapipe')
SUBSET_ROOT = DRIVE_ROOT / f'subsets/top{CLASS_COUNT}'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
GRAPH_REPORT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/report.json'
DRIVE_GRAPH_ROOT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/cache'
RUN_ROOT = SUBSET_ROOT / 'models' / RUN_NAME
TRAINING_CONFIG = RUN_ROOT / 'experiment.pinned.yaml'
LOCAL_REPO = Path('/content/silent-signal')
LOCAL_GRAPH_ROOT = Path('/content/kaggle-vsl-graph-cache')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
for required in (MANIFEST, LABELS, SELECTION, GRAPH_REPORT):
    if not required.is_file():
        raise FileNotFoundError(f'Run notebook 11 first; missing: {required}')
print('Run output:', RUN_ROOT)


In [ ]:
#@title Checkout code and install training dependencies
import subprocess, sys
if not (LOCAL_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
        'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF, f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{LOCAL_REPO}[training]'], check=True)
PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', PROJECT_COMMIT)


In [ ]:
#@title Verify GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title Copy graph cache from Drive to local disk for faster epochs — resumable
import shutil, time
source_files = sorted(DRIVE_GRAPH_ROOT.rglob('*.npz'))
if not source_files:
    raise RuntimeError(f'No graph cache found in {DRIVE_GRAPH_ROOT}; finish notebook 11 first.')
if COPY_GRAPH_CACHE_TO_LOCAL:
    copied = skipped = 0
    started = time.perf_counter()
    for position, source in enumerate(source_files, start=1):
        relative = source.relative_to(DRIVE_GRAPH_ROOT)
        destination = LOCAL_GRAPH_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size == source.stat().st_size:
            skipped += 1
        else:
            shutil.copy2(source, destination)
            copied += 1
        if position == 1 or position == len(source_files) or position % 500 == 0:
            elapsed = time.perf_counter() - started
            rate = position / elapsed if elapsed else 0
            eta = (len(source_files) - position) / rate / 60 if rate else 0
            print(f'{position:,}/{len(source_files):,} copied={copied:,} skipped={skipped:,} ETA={eta:.1f} min')
    GRAPH_ROOT = LOCAL_GRAPH_ROOT
else:
    GRAPH_ROOT = DRIVE_GRAPH_ROOT
print('Training graph root:', GRAPH_ROOT)


In [ ]:
#@title Pin the exact architecture, data identity, and anti-overfit settings
import hashlib, json, yaml
graph_report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
selection = json.loads(SELECTION.read_text(encoding='utf-8'))
manifest_sha256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
payload = {
    'schema_version': 1,
    'evaluation_protocol': selection['split_protocol'],
    'model': {
        'architecture': 'graph_spatial_temporal_transformer_v1',
        'input_dim': 7, 'num_nodes': 75, 'max_frames': 64,
        'hidden_dim': 64, 'embedding_dim': 64, 'num_blocks': 1,
        'temporal_kernel': 3, 'spatial_layers': 1, 'temporal_layers': 1,
        'num_heads': 4, 'ffn_dim': 128, 'dropout': DROPOUT,
        'num_classes': CLASS_COUNT,
    },
    'smoke': {
        'batch_size': 4, 'train_steps': 1, 'sample_limit': 8,
        'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    },
    'training': {
        'batch_size': BATCH_SIZE, 'max_epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING, 'gradient_clip': 1.0,
        'early_stopping_min_epochs': EARLY_STOPPING_MIN_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta': EARLY_STOPPING_MIN_DELTA,
        'num_workers': 0, 'seed': SEED,
        'coordinate_scale_jitter': 0.10,
        'coordinate_translation_jitter': 0.03,
        'joint_dropout': 0.05,
    },
    'expected': {
        'manifest_sha256': manifest_sha256,
        'preprocessing_fingerprint': graph_report['preprocessing_fingerprint'],
    },
}
TRAINING_CONFIG.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
assert payload['model']['architecture'] == 'graph_spatial_temporal_transformer_v1'
print(TRAINING_CONFIG.read_text(encoding='utf-8'))


In [ ]:
#@title Train with validation logging, resume, and early stopping
import subprocess, sys
command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.train_pose_graph',
    '--config', str(TRAINING_CONFIG), '--manifest', str(MANIFEST),
    '--labels', str(LABELS), '--graph-root', str(GRAPH_ROOT),
    '--output-root', str(RUN_ROOT), '--device', 'cuda',
    '--progress-every', '20', '--project-commit', PROJECT_COMMIT,
]
if RESUME:
    command.append('--resume')
if RUN_TEST_AFTER_TRAINING:
    command.append('--run-test')
print('+', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
#@title Show the final report and comparison plots
import json
from IPython.display import Image, display
report = json.loads((RUN_ROOT / 'training_report.json').read_text(encoding='utf-8'))
summary = {
    'architecture': report['architecture'],
    'completed_epochs': report['completed_epochs'],
    'best_epoch': report['best_epoch'],
    'early_stopping': report['early_stopping'],
    'generalization_gap': report['best_epoch_generalization_gap'],
    'evaluation': {
        split: {key: values[key] for key in ('loss', 'top1', 'macro_precision', 'macro_recall', 'macro_f1')}
        for split, values in report['evaluation'].items()
    },
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
for filename in (
    'training_curves.png', 'split_metrics_comparison.png',
    'test_confusion_matrix.png', 'test_per_class_f1.png',
):
    path = RUN_ROOT / filename
    if path.is_file():
        display(Image(filename=str(path)))


## Reading overfitting

Use `training_curves.png`: overfitting begins when training loss keeps falling while validation loss rises, with a widening train–validation Top-1/F1 gap. The red dashed line is the checkpoint selected by minimum validation loss. `MAX_EPOCHS` is only a ceiling; patience-based early stopping ends training and restores that best checkpoint.
